# Logprob Metric Distributions by Text Length

Recompute token-derived metrics using the first N tokens of each text and compare HG/MG distributions in 3D. Metrics are calculated from stored stage-1 top-20 logprobs; no model inference is performed.

In [ ]:
from __future__ import annotations

import os
from IPython.display import display
from pathlib import Path

import seaborn as sns

from notebooks.utils.logprob_metrics_distributions_utils import (
    TOKEN_DERIVED_METRICS,
    compute_length_metric_table,
    default_length_metrics_path,
    load_length_metric_table,
    plot_length_metric_distributions,
    save_length_metric_table,
)

sns.set_theme(style="whitegrid", context="notebook")

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"
PLOT_OUTPUT_ROOT = DATA_ROOT / "plots" / "metric_length_distribution"
PLOT_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## Parameters

Only complete prefixes are evaluated. For example, with `TOKEN_LENGTH_STEP = 50`, a 120-token text contributes lengths 50 and 100.

In [ ]:
selected_datasets = None
selected_models = None
selected_models = [
    "gemma_3_1b_it",
    "gemma_3_270m_it",
    "littlelamb",
    "smollm2_360m",
]
TOKEN_LENGTH_STEP = 50
MAX_ROWS_PER_DATASET_SPLIT = 500
SOURCE_BATCH_SIZE = 256
HISTOGRAM_BINS = 10
RECOMPUTE = False
LENGTH_METRICS_PATH = default_length_metrics_path(PROCESSED_ROOT)

## Recompute Truncated Metrics

In [ ]:
if RECOMPUTE or not LENGTH_METRICS_PATH.exists():
    length_metric_df = compute_length_metric_table(
        PROCESSED_ROOT,
        step=TOKEN_LENGTH_STEP,
        selected_datasets=selected_datasets,
        selected_models=selected_models,
        batch_size=SOURCE_BATCH_SIZE,
        max_rows_per_dataset_split=MAX_ROWS_PER_DATASET_SPLIT,
        show_progress=True,
    )
    save_length_metric_table(length_metric_df, LENGTH_METRICS_PATH)
else:
    length_metric_df = load_length_metric_table(LENGTH_METRICS_PATH)

print(f"Stored at: {LENGTH_METRICS_PATH}")
display(length_metric_df.head())
print(f"Rows: {len(length_metric_df):,}")
print(f"Lengths: {sorted(length_metric_df['token_length'].unique()) if not length_metric_df.empty else []}")

## 3D Metric Distributions

Each metric gets one figure per dataset/model pair. The depth axis is the prefix length, and HG/MG are overlaid with separate colors.

In [ ]:
for metric in TOKEN_DERIVED_METRICS:
    if metric in length_metric_df and length_metric_df[metric].notna().any():
        plot_length_metric_distributions(
            length_metric_df, metric, bins=HISTOGRAM_BINS, output_dir=PLOT_OUTPUT_ROOT
        )
    else:
        print(f"Skipping {metric}: no finite values")

## Publication and Presentation Figures

The following cells use 2D views for comparing datasets under one scoring LLM. They keep axes comparable and emphasize the change in HG/MG separation at longer prefixes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

PAPER_TOKEN_LENGTHS = [100, 300, 600, 1200]
PAPER_MODELS = selected_models or sorted(length_metric_df["model"].dropna().unique())
PAPER_MODEL = PAPER_MODELS[0]
PAPER_DATASETS = sorted(length_metric_df["dataset"].dropna().unique())
PAPER_METRICS = [metric for metric in TOKEN_DERIVED_METRICS if metric != "num_tokens"]
ALL_DATASETS_LABEL = "all_datasets"
SPLIT_PALETTE = {"HG": "#2166ac", "MG": "#b2182b"}
LENGTH_PALETTE = dict(zip(PAPER_TOKEN_LENGTHS, sns.color_palette("viridis", len(PAPER_TOKEN_LENGTHS))))

paper_df = length_metric_df.loc[
    length_metric_df["model"].eq(PAPER_MODEL)
    & length_metric_df["dataset"].isin(PAPER_DATASETS)
    & length_metric_df["token_length"].isin(PAPER_TOKEN_LENGTHS)
].copy()
PAPER_TOKEN_LENGTHS = [length for length in PAPER_TOKEN_LENGTHS if length in paper_df["token_length"].unique()]
print(f"Visualizing model: {PAPER_MODEL}")
print(f"Datasets: {', '.join(PAPER_DATASETS)}")
print(f"Token lengths: {PAPER_TOKEN_LENGTHS}")

### Faceted KDE distributions

One figure is produced per metric. Columns are datasets, rows are prefix lengths, and colors represent HG/MG. The shared axes make cross-dataset comparison direct.

In [ ]:
for metric in PAPER_METRICS:
    metric_df = paper_df.dropna(subset=[metric]).copy()
    if metric_df.empty:
        continue
    grid = sns.FacetGrid(
        metric_df, row="token_length", col="dataset", hue="split",
        row_order=PAPER_TOKEN_LENGTHS, col_order=PAPER_DATASETS,
        hue_order=["HG", "MG"], palette=SPLIT_PALETTE,
        height=1.8, aspect=1.25, margin_titles=True,
        sharex=True, sharey=True, despine=True,
    )
    grid.map_dataframe(
        sns.kdeplot, x=metric, fill=True, common_norm=False,
        alpha=0.24, linewidth=1.4, warn_singular=False,
    )
    grid.map_dataframe(sns.kdeplot, x=metric, common_norm=False, linewidth=1.5, warn_singular=False)
    grid.set_axis_labels(metric, "Density")
    grid.set_titles(col_template="{col_name}", row_template="{row_name} tokens")
    grid.add_legend(title="Source type")
    grid.figure.subplots_adjust(top=0.90)
    grid.figure.suptitle(f"{metric}: {PAPER_MODEL}", fontsize=15, fontweight="bold")
    grid.figure.savefig(PLOT_OUTPUT_ROOT / f"kde_{metric}__{ALL_DATASETS_LABEL}__{PAPER_MODEL}.png", dpi=300, bbox_inches="tight")
    plt.show()

### Split violin distributions

These figures are compact alternatives for a presentation. Each violin is split into HG and MG halves, with quartile markers inside.

In [ ]:
for model in PAPER_MODELS:
    model_df = length_metric_df.loc[length_metric_df["model"].eq(model)].copy()
    for metric in PAPER_METRICS:
        metric_df = model_df.dropna(subset=[metric]).copy()
        figure, axes = plt.subplots(1, len(PAPER_DATASETS), figsize=(4.2 * len(PAPER_DATASETS), 4.4), squeeze=False, constrained_layout=True)
        for axis, dataset in zip(axes[0], PAPER_DATASETS):
            subset = metric_df[metric_df["dataset"].eq(dataset)]
            violin_kwargs = dict(
                data=subset, x="token_length", y=metric, hue="split",
                order=PAPER_TOKEN_LENGTHS, hue_order=["HG", "MG"],
                palette=SPLIT_PALETTE, inner="quart", cut=0,
                linewidth=0.8, density_norm="width", ax=axis,
            )
            if subset["split"].nunique() == 2:
                violin_kwargs["split"] = True
            sns.violinplot(**violin_kwargs)
            axis.set_title(dataset)
            axis.set_xlabel("Prefix length (tokens)")
            axis.set_ylabel(metric)
            if axis.get_legend() is not None:
                axis.get_legend().remove()
        handles = [Line2D([0], [0], color=SPLIT_PALETTE[split], linewidth=7, label=split) for split in ["HG", "MG"]]
        figure.legend(handles=handles, title="Source type", loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.04))
        figure.suptitle(f"{metric}: split distributions by dataset and prefix length ({model})", fontsize=15, fontweight="bold")
        figure.savefig(PLOT_OUTPUT_ROOT / f"violin_{metric}__{ALL_DATASETS_LABEL}__{model}.png", dpi=300, bbox_inches="tight")
        plt.show()

### Ridgeline-style length progression

Each dataset has separate HG and MG columns. Within each panel, colored curves show how the distribution changes as the prefix grows.

In [ ]:
for metric in PAPER_METRICS:
    metric_df = paper_df.dropna(subset=[metric]).copy()
    grid = sns.FacetGrid(
        metric_df, row="dataset", col="split", hue="token_length",
        row_order=PAPER_DATASETS, col_order=["HG", "MG"],
        hue_order=PAPER_TOKEN_LENGTHS, palette=LENGTH_PALETTE,
        height=1.65, aspect=2.4, margin_titles=True, sharex=True, sharey=False,
    )
    grid.map_dataframe(sns.kdeplot, x=metric, fill=False, linewidth=1.8, warn_singular=False)
    grid.set_axis_labels(metric, "Density")
    grid.set_titles(row_template="{row_name}", col_template="{col_name}")
    grid.add_legend(title="Tokens")
    grid.figure.subplots_adjust(top=0.91)
    grid.figure.suptitle(f"{metric}: distribution progression for {PAPER_MODEL}", fontsize=15, fontweight="bold")
    grid.figure.savefig(PLOT_OUTPUT_ROOT / f"ridgeline_{metric}__{ALL_DATASETS_LABEL}__{PAPER_MODEL}.png", dpi=300, bbox_inches="tight")
    plt.show()

### Separation summary heatmap

The cell value is the absolute standardized mean difference between MG and HG. Larger values indicate stronger separation. This is a compact summary, not a replacement for the distribution plots.

In [ ]:
separation_rows = []
for (dataset, token_length), group in paper_df.groupby(["dataset", "token_length"], sort=True):
    hg = group.loc[group["split"].eq("HG"), PAPER_METRICS].apply(pd.to_numeric, errors="coerce")
    mg = group.loc[group["split"].eq("MG"), PAPER_METRICS].apply(pd.to_numeric, errors="coerce")
    for metric in PAPER_METRICS:
        hg_values = hg[metric].dropna()
        mg_values = mg[metric].dropna()
        pooled_std = np.sqrt((hg_values.var(ddof=1) + mg_values.var(ddof=1)) / 2)
        score = abs(mg_values.mean() - hg_values.mean()) / pooled_std if pooled_std > 0 else np.nan
        separation_rows.append({"dataset": dataset, "token_length": token_length, "metric": metric, "separation": score})

separation = pd.DataFrame(separation_rows)
separation["column"] = separation["dataset"] + "\n" + separation["token_length"].astype(str)
heatmap_data = separation.pivot(index="metric", columns="column", values="separation")
column_order = [f"{dataset}\n{length}" for dataset in PAPER_DATASETS for length in PAPER_TOKEN_LENGTHS if f"{dataset}\n{length}" in heatmap_data.columns]
heatmap_data = heatmap_data.reindex(columns=column_order)
figure, axis = plt.subplots(figsize=(max(10, 1.0 * len(column_order)), 4.8), constrained_layout=True)
sns.heatmap(heatmap_data, cmap="mako", annot=True, fmt=".2f", linewidths=0.5, linecolor="white", cbar_kws={"label": "|standardized mean difference|"}, ax=axis)
axis.set_title(f"HG/MG separation by metric, dataset, and prefix length: {PAPER_MODEL}", fontweight="bold")
axis.set_xlabel("Dataset / prefix length")
axis.set_ylabel("Metric")
figure.savefig(PLOT_OUTPUT_ROOT / f"separation__{ALL_DATASETS_LABEL}__{PAPER_MODEL}.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
scatter_df = paper_df[["dataset", "split", "token_length", "prompt_mu_score", "prompt_sigma_score"]].copy()
scatter_df["prompt_mu_score"] = pd.to_numeric(scatter_df["prompt_mu_score"], errors="coerce")
scatter_df["prompt_sigma_score"] = pd.to_numeric(scatter_df["prompt_sigma_score"], errors="coerce")
scatter_df = scatter_df.dropna(subset=["prompt_mu_score", "prompt_sigma_score"])
scatter_df = scatter_df[np.isfinite(scatter_df["prompt_mu_score"]) & np.isfinite(scatter_df["prompt_sigma_score"])]

scatter_grid = sns.FacetGrid(
    scatter_df, row="token_length", col="dataset", hue="split",
    row_order=PAPER_TOKEN_LENGTHS, col_order=PAPER_DATASETS,
    hue_order=["HG", "MG"], palette=SPLIT_PALETTE,
    height=1.8, aspect=1.25, margin_titles=True,
    sharex=True, sharey=True, despine=True,
)
scatter_grid.map_dataframe(
    sns.scatterplot, x="prompt_mu_score", y="prompt_sigma_score",
    s=14, alpha=0.35, linewidth=0,
)
scatter_grid.set_axis_labels("Prompt mean (mu)", "Prompt standard deviation (sigma)")
scatter_grid.set_titles(col_template="{col_name}", row_template="{row_name} tokens")
scatter_grid.add_legend(title="Source type")
scatter_grid.figure.subplots_adjust(top=0.90)
scatter_grid.figure.suptitle(f"Prompt mean/std text scatter: {PAPER_MODEL}", fontsize=15, fontweight="bold")
scatter_grid.figure.savefig(PLOT_OUTPUT_ROOT / f"scatter_prompt_mu_sigma__{ALL_DATASETS_LABEL}__{PAPER_MODEL}.png", dpi=300, bbox_inches="tight")
plt.show()